In [10]:
from time import sleep
from datetime import datetime
import requests
from bs4 import BeautifulSoup
import json

In [60]:
def scrape_reviews(company, from_date, from_page=1, to_page=999999, language="en"):
    
    users = []
    ratings = []
    locations = []
    dates = []
    text = []
    
    language = "www" if language == "en" else language
    date_format = "%Y-%m-%dT%H:%M:%S.%fZ"

    try:
        from_date = datetime.strptime(from_date, date_format)
    except:
        raise AssertionError(f"The date '{from_date}' does NOT match the format '{date_format}'")

    for num_page in range(from_page, to_page + 1):
        print(f"Scraping page {num_page} for {company}...")

        if num_page > 1:
            result = requests.get(f"https://{language}.trustpilot.com/review/{company}?page={num_page}&sort=recency")
        else:
            result = requests.get(f"https://{language}.trustpilot.com/review/{company}?sort=recency")

        if result.status_code != 200:
            print(f"Error {result.status_code} while scraping page {num_page} for {company}. If Error 404, robably no more reviews available.")
            return 0
        
        soup = BeautifulSoup(result.content, 'html.parser')

        users = soup.find_all('span', {'class': 'typography_heading-xxs__QKBS8 typography_appearance-default__AAY17'})
        locations = soup.find_all('div', {'class': 'typography_body-m__xgxZ_ typography_appearance-subtle__8_H2l styles_detailsIcon__Fo_ua'})
        ratings = soup.find_all('div', {'class': 'styles_reviewHeader__iU9Px'})

        dates_html = soup.find_all('div', {'class': 'styles_reviewHeader__iU9Px'})
        dates = [dates_html[i].find("time")["datetime"] for i in range(len(dates_html))]

        # Extracting titles and contents separately
        review_containers = soup.find_all('div', {'class': 'styles_reviewContent__0Q2Tg'})
        for review in review_containers:
            # First part of the review is usually the title
            title = review.find('h2').get_text() if review.find('h2') else "No Title"
            content = review.find('p').get_text() if review.find('p') else "No Content"
            review = title + " " + content
            text.append(review)

        for num_review in range(len(users)):
            full_review = dict()
            full_review["Username"] = users[num_review].get_text()
            full_review["Location"] = locations[num_review].get_text()
            full_review["Rating"] = ratings[num_review]["data-service-review-rating"]
            full_review["Date"] = dates[num_review]
            if datetime.strptime(full_review["Date"],date_format) < from_date:
                print(f"Reached reviews older than {from_date}. Stopping scraping for {company}.")
                return 1
            full_review["Review"] = text[num_review]
            full_review_serialized = json.dumps(full_review).encode('utf-8')
            print(full_review_serialized)

        sleep(5)

    return 1

In [80]:
with open("urls-trustpilot.json", "r") as file:
    cacca = json.load(file)
    company, from_date_str = list(cacca.keys()), list(cacca.values())

In [45]:
company = "zumospin.com"
language = "en"
num_page = 1
language = "www" if language == "en" else language 
result = requests.get(f"https://{language}.trustpilot.com/review/{company}?page={num_page}&sort=recency")
soup = BeautifulSoup(result.content, 'html.parser')

In [8]:
from_date_str = "2024-10-00T00:00:00.000Z"

In [61]:
scrape_reviews(company="zumospin.com", 
               from_date = from_date_str,
               language="en")

Scraping page 1 for zumospin.com...
b'{"Username": "Sabine van Opstal", "Location": "NL", "Rating": "5", "Date": "2024-10-11T21:00:41.000Z", "Review": "The support i get Date of experience: October 11, 2024"}'
b'{"Username": "May Wong", "Location": "NL", "Rating": "5", "Date": "2024-10-11T20:17:38.000Z", "Review": "Very helpful tucker with my problem\\u2026 Very helpful tucker with my problem great site too\\ud83d\\ude42"}'
b'{"Username": "maria daurbekova", "Location": "NL", "Rating": "5", "Date": "2024-10-11T19:03:35.000Z", "Review": "I like zumospin.com  I like zumospin.com ! Good customer service and a lot of free spins / bonuses!!"}'
b'{"Username": "DD", "Location": "NL", "Rating": "5", "Date": "2024-10-09T23:42:57.000Z", "Review": "\\ud83d\\udc4dthe best !!! Date of experience: October 09, 2024"}'
b'{"Username": "G Snoek", "Location": "NL", "Rating": "4", "Date": "2024-10-09T10:14:02.000Z", "Review": "Good casino with some nice extra\\u2019s  A big assortment of games.Nice bonus 

1